# 33 - Spikes state, mirrored rows, move typicality (experiment 16)

The last list of ideas looked for new information rather than refinements, and its three best ideas were first measured on fast models only. This notebook measures
them on the LightGBM of experiment 04 with the R1 + R3 team encoder (the model of submission 27):

- **S, Spikes state** (`hazard_features`, three columns): the moves are replayed in order. Spikes sets the opponent's side and Rapid
  Spin clears the user's side, unless the opponent's active Pokémon is a Ghost type. The 155 features count the Spikes moves but ignore Rapid
  Spin and the order. No target is used. First check: LogReg +0.65 points (15/15 folds) and +0.49 on other seeds; Random Forest +0.42;
- **M, mirrored rows** (`MirrorEstimator`): each training battle is added as seen from the other side, with the opposite label, and
  predictions average the two views. Columns with no counterpart become missing in the mirrored rows. First check: HistGB +0.35 (25/30);
- **L, move typicality** (`MoveTypicalityEncoder`, three columns): how usual the moves of each side are for its species, counted inside the
  folds and leaving each battle out of its own counts. No target is used. First check: LogReg +0.41 (11/15), the best of 7 variants.
- **Combinations:** S + M and S + L. Earlier checks suggest that the gains of S and L may not add up.

**Decision rule (fixed before running)**, paired against the reference, for each candidate:
- seeds 42 / 7 / 123, 15 folds: at least **+0.3 points** and at least **10 of 15 folds** improved;
- new seeds 8 / 9 / 10, never used in this project, 15 folds: at least **+0.15 points**. Only the candidates that pass the first step are
  measured here;
- control: the new columns shuffled across battles must gain less than **+0.2 points** over the reference. M adds no columns and its
  mirrored rows exist only in training, so its check is that mirroring twice gives the original values.

There is no further iteration. The candidates that pass become submissions. Seven comparisons on the same folds make the best one slightly optimistic,
and all these folds resample the same 10,000 battles.

## 0. Setup

In [1]:
import os
import sys
import json
import time
import warnings
from pathlib import Path

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.features import hazard_features, move_key_features, team_key_features
from src.models import MirrorEstimator, get_model_specs, mirror_frame, with_params
from src.move_typicality import KEY_P1_MOVES, KEY_P2_MOVES, MoveTypicalityEncoder, with_move_typicality
from src.team_strength import KEY_P1, KEY_P2, TeamStrengthEncoder, with_team_strength
from src.validation import REPEAT_SEEDS, SEED, cv_scores

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)
PROC = ROOT / "data" / "processed"
OUT = PROC / "models"
OUT.mkdir(parents=True, exist_ok=True)

team, timeline, states = [pd.read_pickle(PROC / f"{name}_train.pkl") for name in ("team", "timeline", "states")]
species = pd.read_pickle(PROC / "species_table.pkl")
ref = pd.read_pickle(PROC / "features_train.pkl")             # v1 features (notebook 03)
with open(PROC / "feature_groups.json") as f:
    groups = json.load(f)
y = ref.pop("player_won")
NUM = [c for c in ref.columns if c not in groups["species_cat"]]
KEYS = [KEY_P1, KEY_P2]
MOVE_KEYS = [KEY_P1_MOVES, KEY_P2_MOVES]
spikes = hazard_features(team, timeline, species).reindex(ref.index)
SPIKES = list(spikes.columns)
X = pd.concat([ref[NUM], team_key_features(team, states).reindex(ref.index), spikes,
               move_key_features(timeline).reindex(ref.index)], axis=1)
assert X[KEYS + MOVE_KEYS + SPIKES].notna().all().all()
print(f"battles: {len(X):,} | features: {len(NUM)} + team keys + {len(SPIKES)} Spikes columns + move keys")

specs = get_model_specs()
LGBM = with_params(specs["lightgbm"], {
    "model__colsample_bytree": 0.6286197472598067, "model__learning_rate": 0.006552992991573526,
    "model__min_child_samples": 106, "model__n_estimators": 1424, "model__num_leaves": 12,
    "model__reg_alpha": 0.6070874110546393, "model__reg_lambda": 2.7129089558470034,
    "model__subsample": 0.6629794526009424})
R1_R3 = {"train_unseen": 0.3, "pairs": True, "per_side": True}       # encoder of submissions 27-29
NEW_SEEDS = (8, 9, 10)


def build(s=False, m=False, l=False):
    # Returns the model and the columns it needs, for a combination of the three ideas
    model = with_team_strength(MirrorEstimator(LGBM) if m else LGBM, **R1_R3)
    columns = NUM + KEYS
    if s:
        columns = columns + SPIKES
    if l:
        model = with_move_typicality(model)
        columns = columns + MOVE_KEYS
    return model, columns


REFERENCE = "reference (submission 27)"
CONFIGS = {REFERENCE: {}, "S Spikes": {"s": True}, "M mirror": {"m": True}, "L moves": {"l": True},
           "S + M": {"s": True, "m": True}, "S + L": {"s": True, "l": True}}
CANDIDATES = [name for name in CONFIGS if name != REFERENCE]


def repeated(fn, seeds=REPEAT_SEEDS):
    return np.concatenate([fn(s) for s in seeds])


def score(config, data=X, seeds=REPEAT_SEEDS):
    model, columns = build(**config)
    return repeated(lambda s: cv_scores(model, data[columns], y, seed=s), seeds)

battles: 10,000 | features: 155 + team keys + 3 Spikes columns + move keys


## 1. A look at the new columns

- **Spikes:** how often each side has them at turn 30, and the win rate of P1 in each case, overall and in the close battles
  (`|diff_hp_total| <= 0.5`). A descriptive view: it does not separate the effect of the Spikes from the skill of the player using them.
- **Move typicality:** the win rate of P1 by quintile of each side's score.

In [2]:
state = pd.DataFrame({"p1 has Spikes": X["p1_spikes_on"] == 1, "p2 has Spikes": X["p2_spikes_on"] == 1})
close = X["diff_hp_total"].abs() <= 0.5
summary = pd.DataFrame({"battles": y.groupby([state["p1 has Spikes"], state["p2 has Spikes"]]).size(),
                        "P1 wins": y.groupby([state["p1 has Spikes"], state["p2 has Spikes"]]).mean(),
                        "close battles": y[close].groupby([state.loc[close, "p1 has Spikes"], state.loc[close, "p2 has Spikes"]]).size(),
                        "P1 wins, close": y[close].groupby([state.loc[close, "p1 has Spikes"], state.loc[close, "p2 has Spikes"]]).mean()})
display(summary.round(3))

largest = X[NUM].corrwith(X["diff_spikes_on"]).abs().sort_values(ascending=False).head(3).round(3).to_dict()
print("correlation of diff_spikes_on with the winner:", round(X["diff_spikes_on"].corr(y), 3), "| largest with the 155 features:", largest)

typical = MoveTypicalityEncoder().fit_transform(X[MOVE_KEYS])
for column in ("p1_move_lp", "p2_move_lp"):
    print(column, "win rate of P1 by quintile:", y.groupby(pd.qcut(typical[column], 5, labels=False)).mean().round(3).tolist())

battles  P1 wins  close battles  P1 wins, close
p1 has Spikes p2 has Spikes                                                 
False         False             2969    0.491            955           0.574
              True              2240    0.600            797           0.693
True          False             2003    0.399            678           0.493
              True              2788    0.501            994           0.607

correlation of diff_spikes_on with the winner: -0.131 | largest with the 155 features: {'p2_n_hazard_moves': 0.308, 'p1_n_hazard_moves': 0.289, 'diff_n_setup_moves': 0.13}
p1_move_lp win rate of P1 by quintile: [0.422, 0.512, 0.55, 0.494, 0.522]
p2_move_lp win rate of P1 by quintile: [0.554, 0.484, 0.484, 0.494, 0.484]


## 2. Checks

- **Mirror:** mirroring twice gives back every column that has a counterpart, and the columns without one are missing.
- **Move typicality, own battle:** on 12 battles, the value from `fit_transform` must equal the value the battle gets when the encoder is
  fitted on all the other battles. So a battle does not count itself.

In [3]:
encoded = TeamStrengthEncoder(**R1_R3).fit_transform(X[NUM + KEYS + SPIKES], y)
twice = mirror_frame(mirror_frame(encoded))
paired = [c for c in encoded.columns if twice[c].notna().all()]
gone = [c for c in encoded.columns if c not in paired]
assert np.allclose(twice[paired].to_numpy(dtype=float), encoded[paired].to_numpy(dtype=float), equal_nan=True)
print(f"mirror: {len(paired)} columns kept, {len(gone)} without a counterpart or with missing values")
print("examples without a counterpart:", gone[:8])

rng = np.random.default_rng(SEED)
for pos in rng.choice(len(X), size=12, replace=False):
    rest = X[MOVE_KEYS].drop(index=X.index[pos])
    alone = MoveTypicalityEncoder().fit(rest).transform(X[MOVE_KEYS].iloc[[pos]])
    assert np.allclose(alone[["p1_move_lp", "p2_move_lp"]].to_numpy(), typical[["p1_move_lp", "p2_move_lp"]].iloc[[pos]].to_numpy(), equal_nan=True)
print("move typicality: a battle does not count itself, on 12 battles")

mirror: 127 columns kept, 39 without a counterpart or with missing values
examples without a counterpart: ['p1_team_mean_base_hp', 'p1_team_mean_base_atk', 'p1_team_mean_base_def', 'p1_team_mean_base_spa', 'p1_team_mean_base_spd', 'p1_team_mean_base_spe', 'p1_team_mean_bst', 'p1_team_max_base_spe']
move typicality: a battle does not count itself, on 12 battles


## 3. Paired CV on seeds 42 / 7 / 123

In [4]:
t0 = time.time()
scores = {}
for name, config in CONFIGS.items():
    scores[name] = score(config)
    print(f"{name:28s} {scores[name].mean():.4f} ({time.time() - t0:.0f}s)")

table = pd.DataFrame({name: {"accuracy": scores[name].mean(),
                             "vs reference (points)": (scores[name] - scores[REFERENCE]).mean() * 100,
                             "folds better": int((scores[name] > scores[REFERENCE]).sum()),
                             "folds worse": int((scores[name] < scores[REFERENCE]).sum())} for name in CONFIGS}).T
table

reference (submission 27)    0.6888 (17s)
S Spikes                     0.6892 (31s)
M mirror                     0.6894 (55s)
L moves                      0.6890 (72s)
S + M                        0.6914 (94s)
S + L                        0.6922 (111s)


,accuracy,vs reference (points),folds better,folds worse
reference (submission 27),0.688800,0.000000,0.0,0.0
S Spikes,0.689233,0.043333,8.0,7.0
M mirror,0.689400,0.060000,8.0,7.0
L moves,0.689033,0.023333,9.0,6.0
S + M,0.691433,0.263333,10.0,4.0
S + L,0.692233,0.343333,10.0,4.0


## 4. Confirmation on new seeds

Only the candidates that pass the first step are measured. The reference is measured on the same folds.

In [5]:
first_step = [name for name in CANDIDATES if table.loc[name, "vs reference (points)"] >= 0.3 and table.loc[name, "folds better"] >= 10]
print("pass the first step:", first_step or "none")
confirm = {}
if first_step:
    new_reference = score({}, seeds=NEW_SEEDS)
    for name in first_step:
        new_scores = score(CONFIGS[name], seeds=NEW_SEEDS)
        confirm[name] = (new_scores - new_reference).mean() * 100
        print(f"{name:12s} new seeds {new_scores.mean():.4f} vs reference {new_reference.mean():.4f} | {confirm[name]:+.2f} points, "
              f"better in {(new_scores > new_reference).sum()}/15 folds")

pass the first step: ['S + L']
S + L        new seeds 0.6897 vs reference 0.6848 | +0.49 points, better in 12/15 folds


## 5. Control: the new columns shuffled across battles

The same permutation for all. A gain here would mean that the score does not come from the content of the columns.

In [6]:
rng = np.random.default_rng(SEED)
order = rng.permutation(len(X))
X_shuffled = X.copy()
X_shuffled[SPIKES] = X[SPIKES].to_numpy()[order]
X_shuffled[MOVE_KEYS] = X[MOVE_KEYS].to_numpy()[order]
shuffled = {}
for name, config in {"S Spikes": CONFIGS["S Spikes"], "L moves": CONFIGS["L moves"]}.items():
    shuffled[name] = (score(config, data=X_shuffled) - scores[REFERENCE]).mean() * 100
    print(f"{name:12s} shuffled: {shuffled[name]:+.2f} points against the reference")

S Spikes     shuffled: -0.13 points against the reference
L moves      shuffled: -0.19 points against the reference


## 6. Decision

For each candidate: at least +0.3 points and 10/15 folds; at least +0.15 points on seeds 8 / 9 / 10; shuffled columns below +0.2 points.

In [7]:
decisions = {}
for name in CANDIDATES:
    config = CONFIGS[name]
    controls = [shuffled[key] < 0.2 for flag, key in (("s", "S Spikes"), ("l", "L moves")) if config.get(flag)]
    checks = {
        "gain >= +0.3": table.loc[name, "vs reference (points)"] >= 0.3,
        "folds >= 10": table.loc[name, "folds better"] >= 10,
        "new seeds >= +0.15": confirm.get(name, -np.inf) >= 0.15,
        "control": all(controls),
    }
    decisions[name] = all(checks.values())
    print(f"{name:10s} " + " | ".join(f"{'OK' if ok else 'NO'} {rule}" for rule, ok in checks.items()) + f" -> {'ADOPT' if decisions[name] else 'do not adopt'}")

table["new seeds vs reference"] = pd.Series(confirm)
table["adopted"] = pd.Series(decisions)
table.to_csv(OUT / "spikes_mirror_moves.csv")

S Spikes   NO gain >= +0.3 | NO folds >= 10 | NO new seeds >= +0.15 | OK control -> do not adopt
M mirror   NO gain >= +0.3 | NO folds >= 10 | NO new seeds >= +0.15 | OK control -> do not adopt
L moves    NO gain >= +0.3 | NO folds >= 10 | NO new seeds >= +0.15 | OK control -> do not adopt
S + M      NO gain >= +0.3 | OK folds >= 10 | NO new seeds >= +0.15 | OK control -> do not adopt
S + L      OK gain >= +0.3 | OK folds >= 10 | OK new seeds >= +0.15 | OK control -> ADOPT


## 7. Summary

| Candidate | Seeds 42/7/123 | vs reference | Folds better / worse | New seeds 8/9/10 |
|---|---|---|---|---|
| Reference (submission 27) | 0.6888 | - | - | 0.6848 |
| S Spikes | 0.6892 | +0.04 | 8 / 7 | - |
| M mirror | 0.6894 | +0.06 | 8 / 7 | - |
| L moves | 0.6890 | +0.02 | 9 / 6 | - |
| S + M | 0.6914 | +0.26 | 10 / 4 | - |
| **S + L** | **0.6922** | **+0.34** | **10 / 4** | **0.6897 (+0.49, 12/15)** |

- **Decision by the rule fixed before running: S + L adopted**, the only candidate to pass. Shuffled columns: S -0.13, L -0.19 points.
- **Fast models overstated the single ideas:** S alone gives +0.04 on LightGBM against +0.65 on LogReg. LightGBM could already approximate the Spikes state from the counts.
- **The gain appears only in combination** (+0.04 and +0.02 alone, +0.34 and +0.49 together). There is no established explanation, and adding columns also changes which ones LightGBM samples, so part of the gain may be noise.
- All these folds resample the same 10,000 battles, so the new seeds are not independent data.
- Descriptive: in close battles (`|diff_hp_total| <= 0.5`) P1 wins 69.3% when only P2 has Spikes on its field and 49.3% when only P1 does. This does not separate the effect of the Spikes from the skill of the player using them.